# Critical Input DEQN: Fixed Taylor Diagnostics

This notebook is a diagnostic lab for the fixed Taylor rule. It checks whether the normal steady branch exists in the trained objects, whether the fixed Taylor network selects that branch, and whether simulated/IRF paths produce a genuine bottleneck mechanism instead of a low-output collapse.

## 0. Setup

Run this notebook after training the natural benchmark and fixed Taylor network. It expects `natural/natural.pt` and `fixed_taylor/fixed.pt` under `baseline_artifacts/critical_input_deqn`.

In [ ]:
# Configure paths, device, and shared imports.
from pathlib import Path
import json
import math
import numpy as np
import torch
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent

BASE = ROOT / 'baseline_artifacts' / 'critical_input_deqn'
NATURAL_PATH = BASE / 'natural' / 'natural.pt'
FIXED_PATH = BASE / 'fixed_taylor' / 'fixed.pt'
POST_OUT = BASE / 'postprocess_fixed_only'

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = 'float64'
dtype = torch.float64 if DTYPE == 'float64' else torch.float32

print('ROOT:', ROOT)
print('DEVICE:', DEVICE)
print('NATURAL_PATH:', NATURAL_PATH, NATURAL_PATH.exists())
print('FIXED_PATH:', FIXED_PATH, FIXED_PATH.exists())

In [ ]:
# Load trained networks and baseline parameters.
from src.critical_input_deqn.config import QMCConfig
from src.critical_input_deqn.experiments import resolve_params, params_from_metadata
from src.critical_input_deqn.postprocess import load_natural, load_rule
from src.critical_input_deqn.qmc import make_qmc_nodes
from src.critical_input_deqn.residuals import natural_residuals, rule_residuals
from src.critical_input_deqn.transforms import decode_natural_outputs, decode_rule_outputs
from src.critical_input_deqn.config import NATURAL_OUTPUT_NAMES, RULE_OUTPUT_NAMES

if not NATURAL_PATH.exists():
    raise FileNotFoundError(f'Missing natural checkpoint: {NATURAL_PATH}')
if not FIXED_PATH.exists():
    raise FileNotFoundError(f'Missing fixed Taylor checkpoint: {FIXED_PATH}')

params, experiment_meta = resolve_params('baseline', None)
natural = load_natural(NATURAL_PATH, device=DEVICE, dtype=dtype)
params = params_from_metadata(natural.metadata, fallback=params)
fixed = load_rule(FIXED_PATH, policy='fixed', device=DEVICE, dtype=dtype)

qmc_cfg = QMCConfig()
nodes = make_qmc_nodes(512, cfg=qmc_cfg, device=DEVICE, dtype=dtype)

print('normal_capacity_slack:', params.normal_capacity_slack)
print('target steady cap pressure M_zero/mbar:', 1.0 / (1.0 + params.normal_capacity_slack))
print('phi_y:', params.phi_y, '(fixed Taylor does not react to output gap if this is zero)')

## 1. Calm-State Network Check

This is the key branch-selection diagnostic. At `D=X=A=0` and `Delta_{t-1}=1`, the fixed Taylor network should stay close to the natural/normal branch. If it returns `Y/Y_n` close to zero, then residual RMS alone is not enough.

In [ ]:
# Build the calm physical state: D=0, X=0, inherited A=0, Delta_prev=1.
def calm_rule_state(batch=1):
    D = torch.zeros(batch, device=DEVICE, dtype=dtype)
    X = torch.zeros_like(D)
    ell_D = torch.full_like(D, float(params.log_bar_lambda_D))
    ell_X = torch.full_like(D, float(params.log_bar_lambda_X))
    log_Z = torch.full_like(D, float(-0.5 * params.sigma_z**2))
    A = torch.zeros_like(D)
    log_Delta_prev = torch.zeros_like(D)
    return torch.stack([D, X, ell_D, ell_X, log_Z, A, log_Delta_prev], dim=-1)

def scalar(x):
    return float(torch.as_tensor(x).detach().cpu().reshape(-1)[0])

z_calm = calm_rule_state()
z_n_calm = z_calm[:, :6]

with torch.no_grad():
    raw_n = natural.net(z_n_calm)
    out_n = decode_natural_outputs(raw_n, NATURAL_OUTPUT_NAMES)
    res_n, drv_n = natural_residuals(z_n_calm, raw_n, natural.net, nodes, params=params, qmc_cfg=qmc_cfg, fb_epsilon=0.0)

    raw_f = fixed.net(z_calm)
    out_f = decode_rule_outputs(raw_f, RULE_OUTPUT_NAMES)
    res_f, drv_f = rule_residuals(z_calm, raw_f, fixed.net, natural.net, nodes, params=params, qmc_cfg=qmc_cfg, fb_epsilon=0.0, policy='fixed')

print('Natural calm outputs')
for k in ['C_n', 'Y_n', 'R_n_real']:
    print(f'{k:24s}', f'{scalar(out_n[k]): .6e}')
print('natural chi:', f'{scalar(drv_n["chi"]): .6e}')
print('natural cap pressure:', f'{scalar(drv_n["M_zero_rent"] / drv_n["mbar"]): .6e}')

print('\nFixed Taylor calm outputs')
for k in ['C', 'Y', 'Pi', 'Q_A', 'S_p', 'F_p']:
    print(f'{k:24s}', f'{scalar(out_f[k]): .6e}')
print('Y/Y_n:', f'{scalar(out_f["Y"] / out_n["Y_n"]): .6e}')
print('output_gap log:', f'{math.log(max(scalar(out_f["Y"] / out_n["Y_n"]), 1e-300)): .6e}')
print('R:', f'{scalar(drv_f["R"]): .6e}')
print('chi:', f'{scalar(drv_f["chi"]): .6e}')
print('I_A:', f'{scalar(drv_f["I_A"]): .6e}')
print('A_next:', f'{scalar(drv_f["A_next"]): .6e}')
print('cap pressure:', f'{scalar(drv_f["M_zero_rent"] / drv_f["mbar"]): .6e}')

print('\nNatural residuals at calm state')
for k, v in res_n.items():
    print(f'{k:24s}', f'{scalar(v.abs()): .6e}')
print('\nFixed residuals at calm state')
for k, v in res_f.items():
    print(f'{k:24s}', f'{scalar(v.abs()): .6e}')

## 2. Training Logs and Validation Diagnostics

This checks whether the training logs improved but does not replace the calm-state and path diagnostics.

In [ ]:
# Load eval JSONs and print high-signal diagnostics.
def load_json(path):
    path = Path(path)
    if not path.exists():
        print('MISSING:', path)
        return None
    with path.open('r', encoding='utf-8') as fh:
        return json.load(fh)

def metric_block(title, data, keys):
    print('\n' + '=' * 90)
    print(title)
    print('=' * 90)
    if data is None:
        return
    for k in keys:
        if k in data:
            print(f'{k:42s} {float(data[k]): .4e}')

natural_eval = load_json(BASE / 'natural' / 'natural_eval.json')
fixed_eval = load_json(BASE / 'fixed_taylor' / 'fixed_eval.json')

metric_block('Natural validation diagnostics', natural_eval, [
    'overall.rms', 'overall.max_abs', 'n_mc.rms', 'n_resource.rms', 'n_euler.rms',
    'exact_cap_solve_error_rel.rms', 'exact_cap_product_scaled.rms',
])
metric_block('Fixed validation diagnostics', fixed_eval, [
    'overall.rms', 'overall.max_abs', 'hh_euler.rms', 'resource.rms', 'price_index.rms',
    'calvo_S.rms', 'calvo_F.rms', 'Q.rms',
    'exact_cap_solve_error_rel.rms', 'exact_cap_product_scaled.rms',
    'exact_repair_projection.rms', 'exact_repair_capacity_excess.rms',
])

In [ ]:
# Plot training logs if they are present.
def plot_train_log(path, title):
    log = load_json(path)
    if log is None:
        return
    if isinstance(log, dict) and 'history' in log:
        rows = log['history']
    elif isinstance(log, list):
        rows = log
    else:
        print('Unknown log format:', path, type(log))
        print(log if isinstance(log, dict) else '')
        return
    if not rows:
        print('Empty log:', path)
        return
    steps = [r.get('step', i) for i, r in enumerate(rows)]
    plt.figure(figsize=(9, 4))
    for key in ['train_rms', 'val_rms', 'val_max_abs', 'val_max']:
        if key in rows[0]:
            plt.plot(steps, [r.get(key, np.nan) for r in rows], label=key)
    plt.yscale('log')
    plt.title(title)
    plt.xlabel('step')
    plt.grid(True, alpha=0.25)
    plt.legend()
    plt.show()

plot_train_log(BASE / 'natural' / 'natural_train_log.json', 'Natural training log')
plot_train_log(BASE / 'fixed_taylor' / 'fixed_train_log.json', 'Fixed Taylor training log')

## 3. Fixed-Only Postprocess

This simulates fixed Taylor paths and deterministic IRFs without requiring BA, discretion, or commitment checkpoints.

In [ ]:
# Run fixed-only postprocess and save artifacts.
from src.critical_input_deqn.sampling import sample_rule_states
from src.critical_input_deqn.postprocess import (
    simulate_rule_episode,
    evaluate_rule_path,
    save_policy_artifacts,
    simulate_rule_ir_scenarios,
    save_ir_artifacts,
)

POST_OUT.mkdir(parents=True, exist_ok=True)
LENGTH = 2000
BATCH_SIZE = 64
SEED = 777
IR_BURNIN = 400
IR_HORIZON = 160
IR_PRESTEPS = 5
IR_RELIEF_LAG = 8

torch.manual_seed(SEED)
z0 = sample_rule_states(BATCH_SIZE, params=params, device=DEVICE, dtype=dtype, seed=SEED)

with torch.no_grad():
    states = simulate_rule_episode(z0, fixed.net, natural.net, policy='fixed', params=params, length=LENGTH)
    states_np, defs_np = evaluate_rule_path(states, policy='fixed', rule_net=fixed.net, natural_net=natural.net, params=params)
save_policy_artifacts(policy='fixed', states_np=states_np, outputs_np=defs_np, out_dir=POST_OUT)

with torch.no_grad():
    labels, ir_states = simulate_rule_ir_scenarios(
        policy='fixed', rule_net=fixed.net, natural_net=natural.net, params=params,
        burnin=IR_BURNIN, horizon=IR_HORIZON, presteps=IR_PRESTEPS,
        relief_lag=IR_RELIEF_LAG, device=DEVICE, dtype=dtype,
    )
    ir_states_np, ir_defs_np = evaluate_rule_path(ir_states, policy='fixed', rule_net=fixed.net, natural_net=natural.net, params=params)
save_ir_artifacts(policy='fixed', labels=labels, states_np=ir_states_np, outputs_np=ir_defs_np, out_dir=POST_OUT)

print('Saved fixed-only postprocess outputs to:', POST_OUT)
for p in sorted(POST_OUT.glob('*')):
    print(' ', p.name)

## 4. Simulated Path Sanity

The crucial failure mode is `Y/Y_n` collapsing toward zero. If that happens, IRFs cannot be interpreted as bottleneck dynamics.

In [ ]:
# Summarize fixed Taylor simulated paths.
def summarize(name, arr):
    x = np.asarray(arr, dtype=float).reshape(-1)
    x = x[np.isfinite(x)]
    if x.size == 0:
        print(f'{name:36s} no finite values')
        return
    print(
        f'{name:36s} mean={x.mean(): .4e} '
        f'p05={np.quantile(x, 0.05): .4e} '
        f'p50={np.quantile(x, 0.50): .4e} '
        f'p95={np.quantile(x, 0.95): .4e} '
        f'max={x.max(): .4e}'
    )

defs_np = dict(np.load(POST_OUT / 'fixed_definitions.npz'))
for k in [
    'Y', 'Y_n', 'output_gap', 'cap_pressure_ratio', 'chi', 'cap_product_scaled',
    'I_A', 'A', 'repair_projection_residual', 'repair_activation_ratio', 'Pi', 'R',
]:
    if k in defs_np:
        summarize(k, defs_np[k])

print('\nRepair regime frequencies')
for k in [
    'repair_lower_corner_indicator', 'repair_interior_indicator',
    'repair_capacity_bound_indicator', 'repair_positive_indicator',
    'repair_active_or_capacity_indicator',
]:
    if k in defs_np:
        print(f'{k:36s} {np.asarray(defs_np[k], dtype=float).mean(): .4f}')

if 'Y' in defs_np and 'Y_n' in defs_np:
    y_ratio = np.asarray(defs_np['Y'], dtype=float) / np.clip(np.asarray(defs_np['Y_n'], dtype=float), 1e-12, None)
    summarize('Y/Y_n', y_ratio)

print('\nPass/fail')
if 'Y' in defs_np and 'Y_n' in defs_np:
    y_med = float(np.nanmedian(y_ratio))
    print(('PASS ' if 0.8 <= y_med <= 1.2 else 'FAIL ') + f'median Y/Y_n in [0.8,1.2], got {y_med:.4e}')
if 'cap_product_scaled' in defs_np:
    cps = float(np.nanmean(np.asarray(defs_np['cap_product_scaled'], dtype=float) ** 2) ** 0.5)
    print(('PASS ' if cps <= 1e-3 else 'FAIL ') + f'cap product scaled RMS <= 1e-3, got {cps:.4e}')
if 'repair_projection_residual' in defs_np:
    rpr = float(np.nanmean(np.asarray(defs_np['repair_projection_residual'], dtype=float) ** 2) ** 0.5)
    print(('PASS ' if rpr <= 1e-5 else 'FAIL ') + f'repair projection RMS <= 1e-5, got {rpr:.4e}')

In [ ]:
# Plot distributions that reveal branch selection.
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
axes = axes.reshape(-1)
plots = [
    ('Y/Y_n', np.asarray(defs_np['Y']) / np.clip(np.asarray(defs_np['Y_n']), 1e-12, None)),
    ('output_gap', defs_np.get('output_gap')),
    ('cap_pressure_ratio', defs_np.get('cap_pressure_ratio')),
    ('chi', defs_np.get('chi')),
    ('I_A', defs_np.get('I_A')),
    ('A', defs_np.get('A')),
]
for ax, (name, arr) in zip(axes, plots):
    if arr is None:
        ax.set_title(name + ' missing')
        continue
    x = np.asarray(arr, dtype=float).reshape(-1)
    x = x[np.isfinite(x)]
    ax.hist(x, bins=60, alpha=0.85)
    ax.set_title(name)
    ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

## 5. Deterministic IRF Sanity

Only interpret IRFs if the no-event path stays near the normal branch. A valid bottleneck IRF should show disruption raising `cap_pressure_ratio`; when it exceeds one, `chi` should become positive by the exact MCP solver.

In [ ]:
# Summarize IRF scenario deviations from no_event.
ir_defs_np = dict(np.load(POST_OUT / 'IR_fixed_definitions.npz', allow_pickle=True))
labels = list(ir_defs_np['labels'])
labels = [str(x) for x in labels]
print('labels:', labels)
base_idx = labels.index('no_event')
vars_to_check = ['pm', 'mbar', 'cap_pressure_ratio', 'chi', 'Pi', 'output_gap', 'I_A', 'A', 'R']

for label in labels:
    if label == 'no_event':
        continue
    j = labels.index(label)
    print(f'\n--- {label} ---')
    for var in vars_to_check:
        if var not in ir_defs_np:
            continue
        path = np.asarray(ir_defs_np[var], dtype=float)[:, j]
        base = np.asarray(ir_defs_np[var], dtype=float)[:, base_idx]
        dev = path - base
        print(
            f'{var:22s} event={path[IR_PRESTEPS]: .4e} '
            f'peak={path.max(): .4e} min_dev={dev.min(): .4e} max_dev={dev.max(): .4e}'
        )

print('\nIRF pass/fail')
for label in ['D_1x', 'D_3x']:
    if label in labels and 'cap_pressure_ratio' in ir_defs_np:
        j = labels.index(label)
        peak = float(np.nanmax(np.asarray(ir_defs_np['cap_pressure_ratio'], dtype=float)[:, j]))
        print(('PASS ' if peak > 1.0 else 'FAIL ') + f'{label} peak cap_pressure_ratio > 1, got {peak:.4e}')
    if label in labels and 'chi' in ir_defs_np:
        j = labels.index(label)
        peak_chi = float(np.nanmax(np.asarray(ir_defs_np['chi'], dtype=float)[:, j]))
        print(('PASS ' if peak_chi > 1e-5 else 'FAIL ') + f'{label} positive chi appears, got {peak_chi:.4e}')

In [ ]:
# Plot core IRFs against no_event.
plot_vars = ['pm', 'mbar', 'cap_pressure_ratio', 'chi', 'output_gap', 'I_A', 'A', 'R']
fig, axes = plt.subplots(4, 2, figsize=(14, 12), sharex=True)
axes = axes.reshape(-1)
t = np.arange(np.asarray(ir_defs_np[plot_vars[0]]).shape[0]) - IR_PRESTEPS
for ax, var in zip(axes, plot_vars):
    if var not in ir_defs_np:
        ax.set_title(var + ' missing')
        continue
    base = np.asarray(ir_defs_np[var], dtype=float)[:, base_idx]
    for label in labels:
        if label == 'no_event':
            continue
        j = labels.index(label)
        path = np.asarray(ir_defs_np[var], dtype=float)[:, j]
        ax.plot(t, path - base, label=label)
    ax.axvline(0, color='k', lw=0.8, alpha=0.4)
    ax.set_title(var + ' deviation from no_event')
    ax.grid(True, alpha=0.25)
axes[-1].legend(loc='best', fontsize=8)
plt.tight_layout()
plt.show()

## 6. Interpretation Rule

Accept fixed Taylor only if: validation residuals are acceptable, `Y/Y_n` stays near the normal branch on no-event/simulated paths, cap and repair exact diagnostics remain near zero, and disruption raises cap pressure enough to produce positive scarcity rent. If `Y/Y_n` collapses, retraining or parametrization changes are needed before any policy comparison.